In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
# I need to hone in on one of the models and save those weights 
# then I can hopefully run PCA to find the voxels where the motor regressor is not the main source of variance
# potentially two models --> 
#   4 delays, look ahead by 10 (codegemma 7b)
#   16 delays, look ahead by 0 (also codegemma 7b)

# probably need the best performing models too ()

In [ ]:
# From chatGPT
import numpy as np

eps = 1e-12

def r2_score(y_true, y_pred, axis=0):
    ss_res = np.sum((y_true - y_pred)**2, axis=0)
    ss_tot = np.sum((y_true - y_true.mean(axis=0, keepdims=True))**2, axis=0)
    return 1.0 - ss_res / (ss_tot + eps)

def decompose_predictions(Xf_val, Xn_val, betaf_full, betan_full, y_val,
                          betaf_reduced=None, do_permutation_test=False,
                          n_perm=1000, random_seed=0):
    """
    Returns dictionary with per-voxel decomposition stats.
    Shapes:
     - Xf_val (T, Ff), Xn_val (T, Fn)
     - betaf_full (Ff, V), betan_full (Fn, V)
     - y_val (T, V)
     - betaf_reduced optional (Ff, V) -> predictions from features-only model
    """
    T = Xf_val.shape[0]
    # Predicted components (T, V)
    yhat_f_full = Xf_val @ betaf_full        # contribution from features (but weights from full model)
    yhat_n_full = Xn_val @ betan_full        # contribution from nuisance
    yhat_full = yhat_f_full + yhat_n_full

    # center them by time (so var/cov match definitions)
    yhat_f_c = yhat_f_full - yhat_f_full.mean(axis=0, keepdims=True)
    yhat_n_c = yhat_n_full - yhat_n_full.mean(axis=0, keepdims=True)
    yhat_c   = yhat_full - yhat_full.mean(axis=0, keepdims=True)

    # variances and covariance (per voxel)
    var_f = np.mean(yhat_f_c**2, axis=0)       # Var(ŷ_f)
    var_n = np.mean(yhat_n_c**2, axis=0)       # Var(ŷ_n)
    var_total = np.mean(yhat_c**2, axis=0)     # Var(ŷ_total)
    cov_fn = np.mean(yhat_f_c * yhat_n_c, axis=0)  # Cov(ŷ_f, ŷ_n)

    # numerical check: var_total ≈ var_f + var_n + 2*cov_fn
    recon = var_f + var_n + 2.0 * cov_fn
    recon_err = var_total - recon

    # fractions of predicted variance
    frac_pred_nuisance = var_n / (var_total + eps)
    frac_pred_features  = var_f / (var_total + eps)
    frac_shared = (2.0 * cov_fn) / (var_total + eps)

    # R^2 on held-out / validation set
    r2_full = r2_score(y_val, yhat_full)

    delta_r2 = None
    r2_features_only = None
    if betaf_reduced is not None:
        yhat_features_reduced = Xf_val @ betaf_reduced
        r2_features_only = r2_score(y_val, yhat_features_reduced)
        delta_r2 = r2_full - r2_features_only

    out = {
        'yhat_f_full': yhat_f_full,   # (T, V)
        'yhat_n_full': yhat_n_full,   # (T, V)
        'yhat_full': yhat_full,       # (T, V)
        'var_f': var_f,               # (V,)
        'var_n': var_n,               # (V,)
        'var_total': var_total,       # (V,)
        'cov_fn': cov_fn,             # (V,)
        'recon_err': recon_err,       # (V,) numerical residual
        'frac_pred_nuisance': frac_pred_nuisance,
        'frac_pred_features': frac_pred_features,
        'frac_shared': frac_shared,
        'r2_full': r2_full,
        'r2_features_only': r2_features_only,
        'delta_r2': delta_r2
    }

    # Optional permutation test: shuffle time in y_val to test how often nuisance fraction exceeds observed
    if do_permutation_test:
        rng = np.random.default_rng(random_seed)
        perm_stats = np.zeros((n_perm, y_val.shape[1]), dtype=np.float64)
        for i in range(n_perm):
            perm_idx = rng.permutation(T)
            # permute the nuisance-predicted timecourse (or permute y_val; here we'll permute Xn_val rows)
            Xn_perm = Xn_val[perm_idx, :]
            yhat_n_perm = Xn_perm @ betan_full
            yhat_n_p_c = yhat_n_perm - yhat_n_perm.mean(axis=0, keepdims=True)
            # compute var of permuted nuisance pred relative to original total predicted variance
            var_n_perm = np.mean(yhat_n_p_c**2, axis=0)
            perm_stats[i, :] = var_n_perm / (var_total + eps)

        # get p-values: fraction of permuted >= observed frac_pred_nuisance
        p_vals = np.mean(perm_stats >= frac_pred_nuisance[np.newaxis, :], axis=0)
        out['perm_pvals_frac_pred_nuisance'] = p_vals
        out['perm_stats_frac_pred_nuisance'] = perm_stats  # only if you want to inspect distribution

    return out


In [ ]:
decomp = decompose_predictions(
    Xf_val=Xf_val,
    Xn_val=Xn_val,
    betaf_full=betaf_full,
    betan_full=betan_full,
    y_val=y_val,
    betaf_reduced=betaf_reduced,     # optional
    do_permutation_test=True,
    n_perm=500
)

# Quick summaries
print("Mean frac predicted variance from nuisance (across voxels):",
      np.nanmean(decomp['frac_pred_nuisance']))

# Top voxels where nuisance drives predictions
top_idx = np.argsort(-decomp['frac_pred_nuisance'])[:20]
print("Top voxels (by frac_pred_nuisance):", top_idx)

# Delta R^2 summary if available
if decomp['delta_r2'] is not None:
    print("Mean ΔR^2 from adding nuisance:", np.nanmean(decomp['delta_r2']))